# Analyzing Trading Strategies with Claude and Multiple Data Sources

In this recipe, we demonstrate how to use Claude's tool use capability to analyze trading strategies by fetching and synthesizing data from multiple sources. Claude autonomously decides which data sources to query, reasons about the data, and produces a structured trade recommendation.

This pattern is useful for any domain where an AI agent needs to:
1. Gather data from heterogeneous APIs
2. Reason about uncertainty across sources
3. Produce a structured recommendation with confidence scoring

We'll build an agent that analyzes prediction markets (e.g., "Will CPI be above 0.3%?") by pulling economic data from FRED, weather forecasts from Open-Meteo, and news sentiment — then synthesizing it all into a trade recommendation.

## Step 1: Set up the environment

Install required libraries and configure the API client. We use `httpx` for HTTP requests to external data APIs.

In [ ]:
%pip install anthropic httpx

In [ ]:
import json
import os
import statistics
from datetime import datetime, timedelta

import httpx
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-haiku-4-5"

## Step 2: Define the data-fetching tools

We define three tools that Claude can call to gather information:

| Tool | Source | Use Case |
|------|--------|----------|
| `fetch_economic_data` | FRED API | CPI, unemployment, GDP, Fed funds rate |
| `fetch_weather_forecast` | Open-Meteo Ensemble API | Temperature forecasts with ensemble spread |
| `analyze_market_sentiment` | News headlines | Sentiment analysis on market topics |

Each tool returns structured data that Claude can reason about. The key design principle: **tools return raw data, not conclusions**. Claude handles the interpretation.

In [ ]:
tools = [
    {
        "name": "fetch_economic_data",
        "description": (
            "Fetches recent economic indicator data from the FRED (Federal Reserve "
            "Economic Data) API. Returns the last several observations with dates "
            "and values, plus computed statistics (mean, trend direction, latest "
            "month-over-month change). Use this for CPI, unemployment rate, GDP "
            "growth, Fed funds rate, and other macroeconomic indicators."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "indicator": {
                    "type": "string",
                    "enum": [
                        "CPI",
                        "unemployment_rate",
                        "GDP",
                        "fed_funds_rate",
                        "consumer_sentiment",
                        "retail_sales",
                        "nonfarm_payrolls",
                    ],
                    "description": "The economic indicator to fetch.",
                }
            },
            "required": ["indicator"],
        },
    },
    {
        "name": "fetch_weather_forecast",
        "description": (
            "Fetches a weather forecast for a given city using the Open-Meteo GFS "
            "ensemble API. Returns the forecast high/low temperature, ensemble "
            "member spread (uncertainty), and the probability that the temperature "
            "will exceed a given threshold. Uses 30 ensemble members for "
            "probabilistic forecasting. Useful for weather-related prediction markets."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": (
                        "City name (e.g., 'Chicago', 'New York', 'Miami'). "
                        "Must be a major US city."
                    ),
                },
                "threshold_temp_f": {
                    "type": "number",
                    "description": (
                        "Optional temperature threshold in Fahrenheit. If provided, "
                        "the response includes the probability of exceeding this value."
                    ),
                },
            },
            "required": ["city"],
        },
    },
    {
        "name": "analyze_market_sentiment",
        "description": (
            "Analyzes recent news sentiment on a given topic by fetching headlines "
            "and categorizing them. Returns a sentiment score (-1.0 to 1.0), headline "
            "count, and sample headlines. Use this to gauge market sentiment around "
            "economic events, policy decisions, or weather events."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": (
                        "The topic to analyze sentiment for "
                        "(e.g., 'CPI inflation February 2026', 'Chicago winter storm')."
                    ),
                }
            },
            "required": ["topic"],
        },
    },
]

## Step 3: Implement the tool functions

Each function fetches real data from public APIs (no API key needed for Open-Meteo). For FRED, we use a demo dataset if no API key is set. For news sentiment, we use simulated headlines to keep the example self-contained.

**Design pattern from production**: These implementations mirror the multi-source edge detection engine in a real trading bot, where each data source is an independent class with caching, error handling, and probability computation.

In [ ]:
# FRED series mapping: indicator name -> (series_id, computation_mode)
FRED_INDICATORS = {
    "CPI": ("CPIAUCSL", "month_over_month_change"),
    "unemployment_rate": ("UNRATE", "level"),
    "GDP": ("A191RL1Q225SBEA", "level"),
    "fed_funds_rate": ("FEDFUNDS", "level"),
    "consumer_sentiment": ("UMCSENT", "level"),
    "retail_sales": ("RSXFS", "month_over_month_change"),
    "nonfarm_payrolls": ("PAYEMS", "month_over_month_change"),
}

# Demo data: used when FRED_API_KEY is not set
DEMO_FRED_DATA = {
    "CPI": {
        "series_id": "CPIAUCSL",
        "indicator": "CPI",
        "unit": "Index (1982-84=100)",
        "observations": [
            {"date": "2026-01-01", "value": 321.5},
            {"date": "2025-12-01", "value": 320.8},
            {"date": "2025-11-01", "value": 319.9},
            {"date": "2025-10-01", "value": 319.3},
            {"date": "2025-09-01", "value": 318.5},
            {"date": "2025-08-01", "value": 317.9},
        ],
        "latest_value": 321.5,
        "recent_mean": 319.65,
        "trend": "rising",
        "mom_changes_pct": [0.22, 0.28, 0.19, 0.25, 0.19],
        "avg_mom_change_pct": 0.226,
        "data_source": "demo",
    },
    "unemployment_rate": {
        "series_id": "UNRATE",
        "indicator": "unemployment_rate",
        "unit": "Percent",
        "observations": [
            {"date": "2026-01-01", "value": 4.1},
            {"date": "2025-12-01", "value": 4.2},
            {"date": "2025-11-01", "value": 4.2},
            {"date": "2025-10-01", "value": 4.1},
            {"date": "2025-09-01", "value": 4.1},
            {"date": "2025-08-01", "value": 4.2},
        ],
        "latest_value": 4.1,
        "recent_mean": 4.15,
        "trend": "stable",
        "data_source": "demo",
    },
    "fed_funds_rate": {
        "series_id": "FEDFUNDS",
        "indicator": "fed_funds_rate",
        "unit": "Percent",
        "observations": [
            {"date": "2026-01-01", "value": 4.33},
            {"date": "2025-12-01", "value": 4.33},
            {"date": "2025-11-01", "value": 4.58},
            {"date": "2025-10-01", "value": 4.83},
            {"date": "2025-09-01", "value": 4.83},
            {"date": "2025-08-01", "value": 5.33},
        ],
        "latest_value": 4.33,
        "recent_mean": 4.705,
        "trend": "falling",
        "data_source": "demo",
    },
}


def fetch_economic_data(indicator: str) -> dict:
    """Fetch economic data from FRED or fall back to demo data."""
    fred_api_key = os.environ.get("FRED_API_KEY", "")

    if not fred_api_key:
        # Use demo data when no API key is available
        if indicator in DEMO_FRED_DATA:
            return DEMO_FRED_DATA[indicator]
        return {
            "error": f"No demo data for '{indicator}'. Set FRED_API_KEY for live data.",
            "available_demo_indicators": list(DEMO_FRED_DATA.keys()),
        }

    if indicator not in FRED_INDICATORS:
        return {"error": f"Unknown indicator: {indicator}"}

    series_id, mode = FRED_INDICATORS[indicator]

    try:
        resp = httpx.get(
            "https://api.stlouisfed.org/fred/series/observations",
            params={
                "series_id": series_id,
                "api_key": fred_api_key,
                "sort_order": "desc",
                "limit": 24,
                "file_type": "json",
            },
            timeout=10.0,
        )
        resp.raise_for_status()
        data = resp.json()
    except httpx.HTTPError as e:
        return {"error": f"FRED API request failed: {e}"}

    # Parse observations
    observations = []
    values = []
    for obs in data.get("observations", []):
        v = obs.get("value", ".")
        if v != ".":
            val = float(v)
            values.append(val)
            observations.append({"date": obs["date"], "value": val})

    if len(values) < 3:
        return {"error": f"Insufficient data for {series_id} (got {len(values)} points)"}

    result = {
        "series_id": series_id,
        "indicator": indicator,
        "observations": observations[:8],
        "latest_value": values[0],
        "recent_mean": round(statistics.mean(values[:6]), 3),
        "trend": "rising" if values[0] > values[2] else "falling" if values[0] < values[2] else "stable",
        "data_source": "FRED API (live)",
    }

    # Compute month-over-month percentage changes for level-change indicators
    if mode == "month_over_month_change" and len(values) >= 4:
        changes = []
        for i in range(len(values) - 1):
            if values[i + 1] != 0:
                changes.append(round((values[i] - values[i + 1]) / values[i + 1] * 100, 3))
        result["mom_changes_pct"] = changes[:6]
        result["avg_mom_change_pct"] = round(statistics.mean(changes[:6]), 3)

    return result

In [ ]:
# City coordinates for weather lookups
CITY_COORDS = {
    "chicago": {"lat": 41.88, "lon": -87.63, "name": "Chicago"},
    "new york": {"lat": 40.71, "lon": -74.01, "name": "New York"},
    "miami": {"lat": 25.76, "lon": -80.19, "name": "Miami"},
    "los angeles": {"lat": 34.05, "lon": -118.24, "name": "Los Angeles"},
    "denver": {"lat": 39.74, "lon": -104.99, "name": "Denver"},
    "houston": {"lat": 29.76, "lon": -95.37, "name": "Houston"},
    "phoenix": {"lat": 33.45, "lon": -112.07, "name": "Phoenix"},
    "seattle": {"lat": 47.61, "lon": -122.33, "name": "Seattle"},
}


def fetch_weather_forecast(city: str, threshold_temp_f: float | None = None) -> dict:
    """Fetch ensemble weather forecast from Open-Meteo (no API key needed)."""
    city_key = city.lower().strip()
    coords = CITY_COORDS.get(city_key)
    if not coords:
        return {
            "error": f"Unknown city: {city}",
            "available_cities": [c["name"] for c in CITY_COORDS.values()],
        }

    tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

    try:
        resp = httpx.get(
            "https://ensemble-api.open-meteo.com/v1/ensemble",
            params={
                "latitude": coords["lat"],
                "longitude": coords["lon"],
                "hourly": "temperature_2m",
                "temperature_unit": "fahrenheit",
                "models": "gfs_seamless",
                "start_date": tomorrow,
                "end_date": tomorrow,
                "timezone": "auto",
            },
            timeout=15.0,
        )
        resp.raise_for_status()
        data = resp.json()
    except httpx.HTTPError as e:
        return {"error": f"Open-Meteo API request failed: {e}"}

    # Extract ensemble member temperatures
    hourly = data.get("hourly", {})
    member_keys = [k for k in hourly if k.startswith("temperature_2m_member")]

    if not member_keys:
        return {"error": "No ensemble members found in response"}

    # Get daily high for each ensemble member (hours 10-18 local = daytime)
    member_highs = []
    for key in member_keys:
        temps = hourly[key]
        daytime = temps[10:19]  # 10 AM to 6 PM local
        if daytime:
            member_highs.append(max(daytime))

    if not member_highs:
        return {"error": "Could not extract daytime temperatures"}

    result = {
        "city": coords["name"],
        "date": tomorrow,
        "ensemble_members": len(member_highs),
        "forecast_high_f": round(statistics.mean(member_highs), 1),
        "min_member_high_f": round(min(member_highs), 1),
        "max_member_high_f": round(max(member_highs), 1),
        "spread_f": round(max(member_highs) - min(member_highs), 1),
        "std_dev_f": round(statistics.stdev(member_highs), 2) if len(member_highs) > 1 else 0,
    }

    # Compute probability of exceeding threshold
    if threshold_temp_f is not None:
        exceeding = sum(1 for t in member_highs if t >= threshold_temp_f)
        prob = exceeding / len(member_highs)
        # Clamp to [0.05, 0.95] — never fully certain
        prob = max(0.05, min(0.95, prob))
        result["threshold_f"] = threshold_temp_f
        result["prob_exceeding_threshold"] = round(prob, 3)
        result["members_exceeding"] = exceeding

    return result

In [ ]:
# Simulated news data for self-contained execution
# In production, you'd use NewsAPI, RSS feeds, or a web scraper
SIMULATED_NEWS = {
    "cpi": {
        "headlines": [
            "Fed officials signal patience on rate cuts as inflation proves sticky",
            "Shelter costs continue to drive CPI higher, economists warn",
            "Used car prices decline for third straight month",
            "Core services inflation remains elevated at 4.8% annualized",
            "Grocery prices stabilize after 18-month surge",
            "Energy prices drop 2.1% month-over-month on mild winter",
        ],
        "overall_sentiment": 0.15,
        "interpretation": "Mixed — shelter hot, goods cooling, services sticky",
    },
    "weather": {
        "headlines": [
            "Arctic blast expected to grip Midwest this weekend",
            "Chicago braces for below-zero wind chills",
            "National Weather Service issues winter storm warning",
            "Heating demand spikes across Great Lakes region",
        ],
        "overall_sentiment": -0.6,
        "interpretation": "Strong cold signal — extreme weather expected",
    },
    "federal reserve": {
        "headlines": [
            "Fed minutes reveal divided committee on pace of cuts",
            "Powell emphasizes data-dependent approach at press conference",
            "Markets price in 60% chance of March rate hold",
            "Regional Fed presidents express concern over services inflation",
            "Treasury yields rise on hawkish Fed commentary",
        ],
        "overall_sentiment": -0.25,
        "interpretation": "Hawkish tilt — rate cuts may be delayed",
    },
}


def analyze_market_sentiment(topic: str) -> dict:
    """Analyze news sentiment for a topic. Uses simulated data for demo."""
    topic_lower = topic.lower()

    # Match topic to simulated data
    matched_key = None
    for key in SIMULATED_NEWS:
        if key in topic_lower:
            matched_key = key
            break

    if matched_key is None:
        # Default response for unmatched topics
        return {
            "topic": topic,
            "headline_count": 0,
            "sentiment_score": 0.0,
            "interpretation": "No relevant headlines found for this topic.",
            "headlines": [],
            "data_source": "simulated",
        }

    news = SIMULATED_NEWS[matched_key]
    return {
        "topic": topic,
        "headline_count": len(news["headlines"]),
        "sentiment_score": news["overall_sentiment"],
        "interpretation": news["interpretation"],
        "headlines": news["headlines"],
        "data_source": "simulated",
    }

Let's verify the tools work before connecting them to Claude. The weather forecast hits a real API.

In [ ]:
# Quick test: fetch live weather data from Open-Meteo
weather_result = fetch_weather_forecast("Chicago", threshold_temp_f=35.0)
print("Weather forecast for Chicago:")
print(json.dumps(weather_result, indent=2))

In [ ]:
# Quick test: fetch economic data (demo mode if no FRED_API_KEY)
econ_result = fetch_economic_data("CPI")
print("CPI data:")
print(json.dumps(econ_result, indent=2))

## Step 4: Build the agentic tool-use loop

This is the core pattern: we send a query to Claude with access to our tools, then loop until Claude finishes reasoning. Claude autonomously decides:
- **Which** tools to call (it may skip irrelevant ones)
- **What arguments** to pass (e.g., which indicator, which city)
- **When to stop** gathering data and produce a recommendation

The loop handles the `tool_use` -> `tool_result` -> continue cycle.

In [ ]:
def process_tool_call(tool_name: str, tool_input: dict) -> str:
    """Route a tool call to the appropriate function and return the result as a string."""
    if tool_name == "fetch_economic_data":
        result = fetch_economic_data(tool_input["indicator"])
    elif tool_name == "fetch_weather_forecast":
        result = fetch_weather_forecast(
            tool_input["city"],
            tool_input.get("threshold_temp_f"),
        )
    elif tool_name == "analyze_market_sentiment":
        result = analyze_market_sentiment(tool_input["topic"])
    else:
        result = {"error": f"Unknown tool: {tool_name}"}

    return json.dumps(result, indent=2)


def analyze_trading_strategy(user_query: str, verbose: bool = True) -> str:
    """Run the agentic loop: Claude decides which data to fetch, then synthesizes."""
    system_prompt = """You are a quantitative trading analyst. When asked to analyze a \
prediction market or trading strategy, you should:

1. Identify which data sources are relevant to the question
2. Fetch data from those sources using the available tools
3. Synthesize the data into a structured analysis
4. Provide a clear recommendation

Your final response MUST be valid JSON with this structure:
{
  "recommendation": "BUY YES" or "BUY NO" or "NO TRADE",
  "confidence": "HIGH" or "MEDIUM" or "LOW",
  "reasoning": "Your detailed reasoning...",
  "edge_estimate_pct": <number>,
  "data_sources_used": ["list of sources"],
  "key_data_points": {"point1": "value1", ...},
  "risks": ["risk1", "risk2", ...]
}

Important rules:
- Only recommend a trade if the edge estimate exceeds 5% (to cover fees and spread)
- Confidence scoring: 1 source = LOW, 2 sources = MEDIUM, 3+ sources = HIGH
- If data is insufficient or contradictory, recommend NO TRADE
- Always identify and list risks"""

    messages = [{"role": "user", "content": user_query}]

    if verbose:
        print(f"{'=' * 60}")
        print(f"Query: {user_query}")
        print(f"{'=' * 60}")

    # Agentic loop — Claude keeps calling tools until it has enough data
    iteration = 0
    max_iterations = 10  # Safety limit

    while iteration < max_iterations:
        iteration += 1

        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )

        if verbose:
            print(f"\n--- Iteration {iteration} (stop_reason: {response.stop_reason}) ---")

        # If Claude is done (no more tool calls), extract final response
        if response.stop_reason == "end_turn":
            final_text = next(
                (block.text for block in response.content if hasattr(block, "text")),
                None,
            )
            if verbose:
                print(f"\nFinal response received after {iteration} iteration(s).")
            return final_text

        # Process all tool calls in this response
        if response.stop_reason == "tool_use":
            # Add assistant's response (with tool_use blocks) to messages
            messages.append({"role": "assistant", "content": response.content})

            # Process each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  Tool call: {block.name}({json.dumps(block.input)})")

                    result = process_tool_call(block.name, block.input)

                    if verbose:
                        # Truncate long results for display
                        display = result[:200] + "..." if len(result) > 200 else result
                        print(f"  Result: {display}")

                    tool_results.append(
                        {
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result,
                        }
                    )

            # Send all tool results back
            messages.append({"role": "user", "content": tool_results})

    return '{"error": "Max iterations reached without a final response"}'

## Step 5: Example 1 — CPI prediction market

This is a real scenario from prediction market trading: the market asks "Will CPI MoM change be above 0.3% this month?" Claude needs to fetch CPI data, check sentiment, and reason about whether the threshold will be exceeded.

In [ ]:
cpi_result = analyze_trading_strategy(
    "Analyze this prediction market: 'Will the next CPI month-over-month change "
    "be above 0.3%?' The market is currently priced at 40 cents (implying 40% "
    "probability). Should I buy YES or NO? Fetch CPI data and news sentiment "
    "to make your recommendation."
)

print("\n" + "=" * 60)
print("STRUCTURED RECOMMENDATION:")
print("=" * 60)
print(cpi_result)

## Step 6: Example 2 — Weather prediction market

Weather prediction markets are another common use case. Here, Claude fetches real ensemble forecast data from Open-Meteo to assess the probability of a temperature threshold being exceeded. The ensemble spread gives a natural measure of forecast uncertainty.

In [ ]:
weather_result = analyze_trading_strategy(
    "Analyze this prediction market: 'Will the high temperature in Chicago "
    "tomorrow exceed 35 degrees Fahrenheit?' The market is priced at 55 cents. "
    "Fetch the weather forecast and any relevant news to make your recommendation."
)

print("\n" + "=" * 60)
print("STRUCTURED RECOMMENDATION:")
print("=" * 60)
print(weather_result)

## Step 7: Example 3 — Multi-source synthesis

The most powerful pattern: Claude gathers data from all three sources and synthesizes a holistic view. This demonstrates the agentic loop at its best — Claude autonomously decides to query multiple endpoints.

In [ ]:
multi_result = analyze_trading_strategy(
    "I'm considering several prediction market positions. Analyze the current "
    "macro environment by checking: (1) the latest CPI trend, (2) Fed funds "
    "rate trajectory, (3) consumer sentiment data, and (4) news sentiment "
    "around the Federal Reserve. Then recommend whether now is a good time to "
    "take positions on 'CPI above 0.3% MoM' (priced at 38 cents) or "
    "'Fed holds rates in March' (priced at 62 cents)."
)

print("\n" + "=" * 60)
print("STRUCTURED RECOMMENDATION:")
print("=" * 60)
print(multi_result)

## Step 8: Parse and display the recommendation

Since Claude returns structured JSON, we can parse it and display it in a more readable format. This is useful for integrating into a trading pipeline or dashboard.

In [ ]:
def display_recommendation(response_text: str) -> None:
    """Parse Claude's JSON response and display it in a readable format."""
    # Extract JSON from the response (Claude may wrap it in markdown code blocks)
    text = response_text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        # Remove first and last lines (code block markers)
        text = "\n".join(lines[1:-1])

    try:
        rec = json.loads(text)
    except json.JSONDecodeError:
        print("Could not parse as JSON. Raw response:")
        print(text)
        return

    # Display as a formatted table
    print(f"Recommendation : {rec.get('recommendation', 'N/A')}")
    print(f"Confidence     : {rec.get('confidence', 'N/A')}")
    print(f"Edge Estimate  : {rec.get('edge_estimate_pct', 'N/A')}%")
    print(f"Data Sources   : {', '.join(rec.get('data_sources_used', []))}")
    print("\nReasoning:")
    print(f"  {rec.get('reasoning', 'N/A')}")

    if rec.get("key_data_points"):
        print("\nKey Data Points:")
        for k, v in rec["key_data_points"].items():
            print(f"  {k}: {v}")

    if rec.get("risks"):
        print("\nRisks:")
        for risk in rec["risks"]:
            print(f"  - {risk}")


# Display the CPI recommendation from Example 1
print("CPI Market Recommendation")
print("-" * 40)
display_recommendation(cpi_result)

## Key Takeaways

This notebook demonstrated several important patterns for building agentic tool-use systems:

1. **Tools return data, not decisions.** Each tool fetches raw data (economic indicators, weather forecasts, news headlines). Claude handles interpretation and synthesis. This separation keeps tools reusable and lets the model reason freely.

2. **The agentic loop handles multi-step reasoning.** Claude may call one tool, reason about the result, then decide to call another. The `while` loop with `stop_reason == "tool_use"` handles this naturally — you don't need to hardcode the sequence of API calls.

3. **Confidence scoring maps to source count.** A simple but effective heuristic: more independent data sources = higher confidence. This prevents overconfident recommendations from a single data point.

4. **Structured JSON output enables downstream integration.** By instructing Claude to return JSON, the recommendation can feed directly into an automated trading pipeline, a dashboard, or a risk management system.

5. **Graceful fallbacks keep the system usable.** Demo data for FRED (when no API key is set) and simulated news data let the notebook run anywhere. In production, you'd add caching, rate limiting, and retry logic to each data source.

### Production Considerations

To use this pattern in a real trading system, you would want to add:
- **Caching** with TTL per data source (e.g., 30 min for weather, 1 hour for economic data)
- **Rate limiting** to respect API quotas (e.g., FRED free tier: 120 requests/minute)
- **Retry logic** with exponential backoff for transient API failures
- **Position sizing** based on edge estimate and Kelly criterion
- **Contradictory position detection** to avoid betting both sides of correlated markets
- **Daily loss limits** and maximum open position caps for risk management